# Linear Regression

**Goal:** Implement linear regression from scratch using the closed-form normal equation and gradient descent in PyTorch, validate against scikit-learn, and understand when each approach is appropriate.

## Configuration

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Imports and Synthetic Dataset

In [2]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Reproducible synthetic dataset
torch.manual_seed(42)
np.random.seed(42)

n, d = 200, 3  # 200 samples, 3 features

# Ground-truth weights and bias
true_w = torch.tensor([2.0, -1.5, 0.8])
true_b = torch.tensor(3.0)

# Feature matrix and targets (CPU for data generation)
X_cpu = torch.randn(n, d)
noise = 0.5 * torch.randn(n)
y_cpu = X_cpu @ true_w + true_b + noise

print(f"Dataset: X shape={X_cpu.shape}, y shape={y_cpu.shape}")
print(f"True weights: {true_w.tolist()}, bias: {true_b.item()}")
print(f"Noise std: {noise.std():.4f}")

Dataset: X shape=torch.Size([200, 3]), y shape=torch.Size([200])
True weights: [2.0, -1.5, 0.800000011920929], bias: 3.0
Noise std: 0.4884


## From Scratch: Closed-Form Normal Equation

The ordinary least squares (OLS) solution solves the normal equations:

$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

We fold the bias into `X` by prepending a column of ones, so `beta = [b, w1, w2, w3]`.

Rather than explicitly inverting `XᵀX` (numerically unstable), we use `torch.linalg.lstsq` which uses a stable decomposition internally.

In [3]:
def ols_normal_equation(X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Solve OLS via the normal equations using a stable least-squares solver.

    Args:
        X: Feature matrix of shape (n, d) — bias column already included.
        y: Target vector of shape (n,).

    Returns:
        Coefficient vector of shape (d,) where beta[0] is bias.
    """
    # torch.linalg.lstsq uses LAPACK's gelsd/gelss — stable even when XᵀX is ill-conditioned
    result = torch.linalg.lstsq(X, y, driver="gelsd")
    return result.solution


# Augment X with bias column (column of ones prepended)
ones = torch.ones(n, 1)
X_aug = torch.cat([ones, X_cpu], dim=1)  # shape (n, 4)

# Solve — lstsq requires float64 on CPU for full driver support
X_aug_f64 = X_aug.double()
y_f64 = y_cpu.double()

beta_ols = ols_normal_equation(X_aug_f64, y_f64).float()

b_ols = beta_ols[0].item()
w_ols = beta_ols[1:]

print("=== Closed-Form OLS Results ===")
print(f"Learned bias:    {b_ols:.4f}  (true: {true_b.item():.4f})")
print(f"Learned weights: {w_ols.tolist()}")
print(f"True weights:    {true_w.tolist()}")

=== Closed-Form OLS Results ===
Learned bias:    2.9908  (true: 3.0000)
Learned weights: [2.0376579761505127, -1.4828875064849854, 0.8221547603607178]
True weights:    [2.0, -1.5, 0.800000011920929]


## From Scratch: Gradient Descent

The MSE gradient with respect to weights `beta` (bias included) is:

$$\nabla_\beta L = \frac{2}{n} X^\top (X\beta - y)$$

We run plain batch gradient descent until convergence.

In [4]:
def gradient_descent_linreg(
    X: torch.Tensor,
    y: torch.Tensor,
    lr: float = 0.05,
    n_iters: int = 2000,
) -> tuple[torch.Tensor, list[float]]:
    """Train linear regression via batch gradient descent.

    Args:
        X: Augmented feature matrix (n, d+1) with bias column.
        y: Target vector (n,).
        lr: Learning rate.
        n_iters: Number of gradient steps.

    Returns:
        Tuple of (beta, loss_history) where beta has shape (d+1,).
    """
    n = X.shape[0]
    beta = torch.zeros(X.shape[1], dtype=X.dtype)
    losses = []

    for i in range(n_iters):
        residuals = X @ beta - y          # (n,)
        loss = (residuals ** 2).mean()    # scalar MSE
        grad = 2.0 / n * (X.T @ residuals)
        beta = beta - lr * grad
        if i % 200 == 0:
            losses.append(loss.item())

    return beta, losses


# Run GD on float32
beta_gd, loss_history = gradient_descent_linreg(X_aug.float(), y_cpu.float(), lr=0.05, n_iters=3000)

b_gd = beta_gd[0].item()
w_gd = beta_gd[1:]

print("=== Gradient Descent Results ===")
print(f"Learned bias:    {b_gd:.4f}  (true: {true_b.item():.4f})")
print(f"Learned weights: {w_gd.tolist()}")
print(f"True weights:    {true_w.tolist()}")
print(f"Final MSE loss:  {loss_history[-1]:.6f}")

=== Gradient Descent Results ===
Learned bias:    2.9908  (true: 3.0000)
Learned weights: [2.037656784057617, -1.482886791229248, 0.8221542835235596]
True weights:    [2.0, -1.5, 0.800000011920929]
Final MSE loss:  0.235057


## Validation Against scikit-learn

We assert that our closed-form OLS solution matches `sklearn.linear_model.LinearRegression` to within `atol=1e-5` (both are exact solvers), and that gradient descent matches to within `atol=1e-3`.

In [5]:
# scikit-learn reference (fits intercept separately)
sk_model = LinearRegression()
sk_model.fit(X_cpu.numpy(), y_cpu.numpy())

sk_w = torch.tensor(sk_model.coef_, dtype=torch.float32)
sk_b = float(sk_model.intercept_)

print("=== scikit-learn Reference ===")
print(f"sklearn bias:    {sk_b:.4f}")
print(f"sklearn weights: {sk_w.tolist()}")

# --- Assertion: OLS matches sklearn (exact solver → tight tolerance) ---
ols_atol = 1e-5
assert abs(b_ols - sk_b) < ols_atol, f"Bias mismatch OLS vs sklearn: {b_ols:.6f} vs {sk_b:.6f}"
assert torch.allclose(w_ols, sk_w, atol=ols_atol), (
    f"Weight mismatch OLS vs sklearn:\n  ours={w_ols}\n  sklearn={sk_w}"
)
print(f"\nOLS matches sklearn (atol={ols_atol}): PASS")

# --- Assertion: GD converges to sklearn ---
atol = 1e-3
assert abs(b_gd - sk_b) < atol, f"Bias mismatch GD vs sklearn: {b_gd:.6f} vs {sk_b:.6f}"
assert torch.allclose(w_gd, sk_w, atol=atol), (
    f"Weight mismatch GD vs sklearn:\n  ours={w_gd}\n  sklearn={sk_w}"
)
print(f"GD converges to sklearn (atol={atol}): PASS")

=== scikit-learn Reference ===
sklearn bias:    2.9908
sklearn weights: [2.0376572608947754, -1.4828872680664062, 0.8221549987792969]

OLS matches sklearn (atol=1e-05): PASS
GD converges to sklearn (atol=0.001): PASS


## R² Score and Predictions

In [6]:
def predict(X_aug: torch.Tensor, beta: torch.Tensor) -> torch.Tensor:
    return X_aug @ beta


y_pred_ols = predict(X_aug.float(), beta_ols)
y_pred_gd  = predict(X_aug.float(), beta_gd)
y_pred_sk  = torch.tensor(sk_model.predict(X_cpu.numpy()), dtype=torch.float32)

def r_squared(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    ss_res = ((y_true - y_pred) ** 2).sum()
    ss_tot = ((y_true - y_true.mean()) ** 2).sum()
    return 1.0 - (ss_res / ss_tot).item()

r2_ols = r_squared(y_cpu, y_pred_ols)
r2_gd  = r_squared(y_cpu, y_pred_gd)
r2_sk  = r_squared(y_cpu, y_pred_sk)

print(f"R² (OLS from scratch): {r2_ols:.6f}")
print(f"R² (GD from scratch):  {r2_gd:.6f}")
print(f"R² (sklearn):          {r2_sk:.6f}")

R² (OLS from scratch): 0.964690
R² (GD from scratch):  0.964690
R² (sklearn):          0.964690


## Plots

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: predicted vs actual
ax = axes[0]
ax.scatter(y_cpu.numpy(), y_pred_ols.detach().numpy(), alpha=0.4, s=15, label="OLS")
ax.scatter(y_cpu.numpy(), y_pred_gd.detach().numpy(),  alpha=0.4, s=15, marker="x", label="GD", color="orange")
lo, hi = y_cpu.min().item(), y_cpu.max().item()
ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="Perfect fit")
ax.set_xlabel("Actual y")
ax.set_ylabel("Predicted y")
ax.set_title(f"Predicted vs Actual  (R²={r2_ols:.4f})")
ax.legend()

# Right: GD loss curve
ax2 = axes[1]
ax2.plot(loss_history, marker="o", markersize=4)
ax2.set_xlabel("Iteration (×200)")
ax2.set_ylabel("MSE Loss")
ax2.set_title("Gradient Descent Training Curve")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("linear_regression_plots.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved.")

Plot saved.


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_74792/3271913409.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Idiomatic Way: scikit-learn

In production, use `sklearn.linear_model.LinearRegression` for small-to-medium tabular data, or `torch.nn.Linear` + an optimizer for large-scale or GPU-accelerated settings.

In [8]:
# scikit-learn — idiomatic
from sklearn.linear_model import LinearRegression as SkLinReg

sk = SkLinReg()
sk.fit(X_cpu.numpy(), y_cpu.numpy())

y_sk = sk.predict(X_cpu.numpy())
print(f"sklearn R²:      {r2_score(y_cpu.numpy(), y_sk):.6f}")
print(f"sklearn coef:    {sk.coef_}")
print(f"sklearn intercept: {sk.intercept_:.6f}")

# PyTorch nn.Linear — idiomatic
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
model = nn.Linear(d, 1).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

X_dev = X_cpu.to(device)
y_dev = y_cpu.unsqueeze(1).to(device)

for _ in range(2000):
    optimizer.zero_grad()
    loss = loss_fn(model(X_dev), y_dev)
    loss.backward()
    optimizer.step()

w_nn = model.weight.data.squeeze().cpu()
b_nn = model.bias.data.item()
print(f"\ntorch nn.Linear weights: {w_nn.tolist()}")
print(f"torch nn.Linear bias:    {b_nn:.4f}")

sklearn R²:      0.964690
sklearn coef:    [ 2.0376573 -1.4828873  0.822155 ]
sklearn intercept: 2.990801



torch nn.Linear weights: [2.0376579761505127, -1.4828875064849854, 0.822154700756073]
torch nn.Linear bias:    2.9908


## Takeaways

- **Normal equation** (`torch.linalg.lstsq`) gives the exact OLS solution in one shot — ideal when `n` and `d` are small enough that a linear solve is cheap (roughly `d` up to a few thousand).
- **Gradient descent** is iterative, requires tuning a learning rate, and converges more slowly, but scales to large datasets and can be extended to regularization, mini-batching, and non-linear objectives trivially.
- Both approaches converge to the same coefficients on well-conditioned data; the difference is computational, not statistical.
- **R²** close to 1 indicates the model explains most of the variance; residual plots (not shown) should also be checked for heteroskedasticity.
- Avoid explicitly computing `(XᵀX)⁻¹`; stable solvers (`lstsq`, `torch.linalg.solve`) are more numerically robust.
- Linear regression connects to `loss-functions` (MSE objective), `gradient-descent`, and `l1-l2-regularization` (ridge adds `λI` to `XᵀX` before solving).